# World Commodities (Yahoo Finance)

In [1]:
from typing import cast
from pathlib import Path

import pandas as pd
import requests
import yfinance as yf
import mgplot as mg

In [2]:
CHART_DIR = "./CHARTS/Commodities/"
mg.set_chart_dir(CHART_DIR)
mg.clear_chart_dir()
SHOW = False

In [3]:
# --- Fetch WTI & Brent ---
tickers = {"CL=F": "WTI (US)", "BZ=F": "Brent (Europe)"}
frames = {}
for ticker, label in tickers.items():
    df = yf.download(ticker, start="2025-12-01", auto_adjust=True, progress=False)
    if df is not None:
        frames[label] = df["Close"].squeeze()

wti_brent = pd.DataFrame(frames)
wti_brent.index = cast(pd.DatetimeIndex, wti_brent.index).to_period("D")

print(f"Date range: {wti_brent.index[0]} to {wti_brent.index[-1]}")
for col in wti_brent.columns:
    print(f"  {col:20s}  min={wti_brent[col].min():.2f}  max={wti_brent[col].max():.2f}")

Date range: 2025-12-01 to 2026-03-20
  WTI (US)              min=55.27  max=98.71
  Brent (Europe)        min=58.92  max=108.65


In [4]:
# --- Chart: WTI vs Brent ---
mg.line_plot_finalise(
    wti_brent,
    title="Crude Oil Prices: WTI vs Brent",
    ylabel="USD per barrel",
    xlabel=None,
    legend={"loc": "best", "fontsize": "x-small"},
    annotate=True,
    axvline={
        "x": pd.Period("2026-02-28", freq="D").ordinal,
        "color": "grey",
        "linestyle": "--",
        "linewidth": 1,
        "label": "Hormuz closure",
    },
    lfooter="NYMEX (WTI) and ICE (Brent) daily closing prices",
    rfooter="Source: Yahoo Finance",
    show=SHOW,
)

In [5]:
# --- Fetch DME Oman & Dubai from Oilprice.com (cached) ---
CACHE_DIR = Path("./CACHE_SOURCE/")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

OILPRICE_BLENDS = {
    "Oman (Middle East)": {"blend_id": 48, "cache": "dme_oman.csv"},
    "Dubai (Middle East)": {"blend_id": 144, "cache": "dubai.csv"},
}


def _fetch_oilprice(blend_id: int, period: int) -> pd.DataFrame:
    """Fetch crude oil prices from Oilprice.com API."""
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36"
        ),
        "X-Requested-With": "XMLHttpRequest",
        "Referer": "https://oilprice.com/oil-price-charts/",
    }
    csrf = requests.get(
        "https://oilprice.com/ajax/csrf", headers=headers, timeout=15
    ).json()
    resp = requests.post(
        "https://oilprice.com/freewidgets/json_get_oilprices",
        headers=headers,
        data={"blend_id": blend_id, "period": period, csrf["name"]: csrf["hash"]},
        timeout=15,
    )
    resp.raise_for_status()
    records = resp.json()["prices"]
    df = pd.DataFrame(records)
    df["date"] = pd.to_datetime(df["time"], unit="s").dt.normalize()
    df["price"] = df["price"].astype(float)
    # Drop stale artifact entries (timestamps far out of sequence)
    df = df.sort_values("date").reset_index(drop=True)
    if len(df) > 1:
        df = df[df["date"] >= df["date"].iloc[-1] - pd.Timedelta(days=400)]
    return df[["date", "price"]]


oilprice_series = {}
for name, cfg in OILPRICE_BLENDS.items():
    cache_file = CACHE_DIR / cfg["cache"]
    if cache_file.exists():
        cached = pd.read_csv(cache_file, parse_dates=["date"])
        fresh = _fetch_oilprice(cfg["blend_id"], period=6)
        combined = pd.concat([cached, fresh]).drop_duplicates(subset="date", keep="last")
        df = combined.sort_values("date").reset_index(drop=True)
        print(f"{name}: updated cache ({len(cached)} -> {len(df)} rows)")
    else:
        df = _fetch_oilprice(cfg["blend_id"], period=5)
        print(f"{name}: initial cache ({len(df)} rows)")
    df.to_csv(cache_file, index=False)
    oilprice_series[name] = pd.Series(
        df["price"].values,
        index=pd.PeriodIndex(df["date"], freq="D"),
        name=name,
    )

Oman (Middle East): updated cache (188 -> 188 rows)


Dubai (Middle East): updated cache (202 -> 202 rows)


In [6]:
# --- Combine all benchmarks ---
prices = wti_brent.copy()
for s in oilprice_series.values():
    prices = prices.join(s, how="outer")
prices = prices.sort_index()
prices = prices.loc[prices.index >= pd.Period("2025-12-01", freq="D")]

print(f"Date range: {prices.index[0]} to {prices.index[-1]}")
for col in prices.columns:
    valid = prices[col].dropna()
    print(f"  {col:20s}  min={valid.min():.2f}  max={valid.max():.2f}  n={len(valid)}")

Date range: 2025-12-01 to 2026-03-20
  WTI (US)              min=55.27  max=98.71  n=76
  Brent (Europe)        min=58.92  max=108.65  n=76
  Oman (Middle East)    min=58.36  max=162.53  n=55
  Dubai (Middle East)   min=58.35  max=137.82  n=56


In [7]:
# --- Chart: All Crude Oil Benchmarks ---
last_dates = {col: prices[col].dropna().index[-1] for col in prices.columns}
lfooter = "Last: " + ", ".join(f"{col.split(' ')[0]} {d}" for col, d in last_dates.items())

mg.line_plot_finalise(
    prices,
    title="Crude Oil Benchmarks",
    ylabel="USD per barrel",
    xlabel=None,
    legend={"loc": "best", "fontsize": "x-small"},
    annotate=True,
    axvline={
        "x": pd.Period("2026-02-28", freq="D").ordinal,
        "color": "grey",
        "linestyle": "--",
        "linewidth": 1,
        "label": "Hormuz closure",
    },
    lfooter=lfooter,
    rfooter="Source: Yahoo Finance / Oilprice.com",
    show=SHOW,
)

print(f"\nChart saved to {CHART_DIR}")


Chart saved to ./CHARTS/Commodities/


In [8]:
# --- Fetch gold data (last 2 years) ---
gold_df = yf.download("GC=F", start="2024-03-20", auto_adjust=True, progress=False)
if gold_df is not None:
    gold = gold_df["Close"].squeeze().dropna()
    gold.index = cast(pd.DatetimeIndex, gold.index).to_period("D")

print(f"Gold date range: {gold.index[0]} to {gold.index[-1]}")
print(f"  Gold  min={gold.min():.2f}  max={gold.max():.2f}")

Gold date range: 2024-03-20 to 2026-03-20
  Gold  min=2157.90  max=5318.40


In [9]:
# --- Chart 2: Gold Price ---
mg.line_plot_finalise(
    gold,
    title="Gold Price",
    ylabel="USD per troy ounce",
    xlabel=None,
    annotate=True,
    lfooter="COMEX daily closing price",
    rfooter="Source: Yahoo Finance",
    show=SHOW,
)

In [10]:
# --- Fetch copper data (last 2 years) ---
copper_df = yf.download("HG=F", start="2024-03-20", auto_adjust=True, progress=False)
if copper_df is not None:
    copper = copper_df["Close"].squeeze().dropna()
    copper.index = cast(pd.DatetimeIndex, copper.index).to_period("D")

print(f"Copper date range: {copper.index[0]} to {copper.index[-1]}")
print(f"  Copper  min={copper.min():.4f}  max={copper.max():.4f}")

Copper date range: 2024-03-20 to 2026-03-20
  Copper  min=3.9375  max=6.1755


In [11]:
# --- Chart 3: Copper Price ---
mg.line_plot_finalise(
    copper,
    title="Copper Price",
    ylabel="USD per pound",
    xlabel=None,
    annotate=True,
    lfooter="COMEX daily closing price",
    rfooter="Source: Yahoo Finance",
    show=SHOW,
)

In [12]:
# --- Fetch natural gas data ---
gas_tickers = {
    "NG=F": "Henry Hub (US)",
    "TTF=F": "TTF (Europe)",
    "JKM=F": "JKM (Asia)",
}
gas_frames = {}
for ticker, label in gas_tickers.items():
    df = yf.download(ticker, start="2025-12-01", auto_adjust=True, progress=False)
    if df is not None:
        s = df["Close"].squeeze().dropna()
        s.index = cast(pd.DatetimeIndex, s.index).to_period("D")
        gas_frames[label] = s

# Fetch EUR/USD rate to convert TTF from EUR/MWh to USD/MMBtu
fx_df = yf.download("EURUSD=X", start="2025-12-01", auto_adjust=True, progress=False)
if fx_df is not None:
    eurusd = fx_df["Close"].squeeze().dropna()
    eurusd.index = cast(pd.DatetimeIndex, eurusd.index).to_period("D")

gas_prices = pd.DataFrame(gas_frames).dropna()

# Convert TTF: EUR/MWh -> USD/MMBtu (1 MWh = 3.412 MMBtu)
MMBTU_PER_MWH = 3.412
ttf_aligned = eurusd.reindex(gas_prices.index, method="ffill")
gas_prices["TTF (Europe)"] = gas_prices["TTF (Europe)"] * ttf_aligned / MMBTU_PER_MWH

print(f"Natural Gas date range: {gas_prices.index[0]} to {gas_prices.index[-1]}")
for col in gas_prices.columns:
    print(f"  {col:20s}  min={gas_prices[col].min():.3f}  max={gas_prices[col].max():.3f}")

Natural Gas date range: 2025-12-01 to 2026-03-19
  Henry Hub (US)        min=2.827  max=7.460
  TTF (Europe)          min=9.066  max=20.783
  JKM (Asia)            min=9.455  max=22.350


In [13]:
# --- Chart 4: Natural Gas Benchmarks ---
mg.line_plot_finalise(
    gas_prices,
    title="Natural Gas Benchmarks",
    ylabel="USD per MMBtu",
    xlabel=None,
    legend={"loc": "best", "fontsize": "x-small"},
    annotate=True,
    axvline={
        "x": pd.Period("2026-02-28", freq="D").ordinal,
        "color": "grey",
        "linestyle": "--",
        "linewidth": 1,
        "label": "Hormuz closure",
    },
    lfooter="TTF converted from EUR/MWh using daily EUR/USD rate",
    rfooter="Source: Yahoo Finance",
    show=SHOW,
)

In [14]:
# --- Fetch grains data (last 2 years) ---
grain_tickers = {"ZW=F": "Wheat", "ZC=F": "Corn"}
grain_frames = {}
for ticker, label in grain_tickers.items():
    df = yf.download(ticker, start="2024-03-20", auto_adjust=True, progress=False)
    if df is not None:
        s = df["Close"].squeeze().dropna()
        s.index = cast(pd.DatetimeIndex, s.index).to_period("D")
        grain_frames[label] = s

grains = pd.DataFrame(grain_frames).dropna()

print(f"Grains date range: {grains.index[0]} to {grains.index[-1]}")
for col in grains.columns:
    print(f"  {col:6s}  min={grains[col].min():.2f}  max={grains[col].max():.2f}")

Grains date range: 2024-03-20 to 2026-03-20
  Wheat   min=495.00  max=700.25
  Corn    min=362.00  max=502.00


In [15]:
# --- Chart 5: Grain Prices ---
mg.line_plot_finalise(
    grains,
    title="Grain Prices: Wheat and Corn",
    ylabel="USD cents per bushel",
    xlabel=None,
    legend={"loc": "best", "fontsize": "x-small"},
    annotate=True,
    lfooter="CBOT daily closing prices",
    rfooter="Source: Yahoo Finance",
    show=SHOW,
)

In [16]:
# --- Fetch urea data ---
urea_df = yf.download("UFV=F", start="2025-12-01", auto_adjust=True, progress=False)
if urea_df is not None:
    urea = urea_df["Close"].squeeze().dropna()
    urea.index = cast(pd.DatetimeIndex, urea.index).to_period("D")

print(f"Urea date range: {urea.index[0]} to {urea.index[-1]}")
print(f"  Urea  min={urea.min():.2f}  max={urea.max():.2f}")

Urea date range: 2025-12-01 to 2026-03-19
  Urea  min=352.25  max=616.50


In [17]:
# --- Chart 6: Urea Price ---
mg.line_plot_finalise(
    urea,
    title="Urea Price",
    ylabel="USD per tonne",
    xlabel=None,
    annotate=True,
    axvline={
        "x": pd.Period("2026-02-28", freq="D").ordinal,
        "color": "grey",
        "linestyle": "--",
        "linewidth": 1,
        "label": "Hormuz closure",
    },
    lfooter="CME Urea (Granular) FOB US Gulf futures daily closing price",
    rfooter="Source: Yahoo Finance",
    show=SHOW,
)

In [18]:
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda

Last updated: 2026-03-21 09:17:23

Python implementation: CPython
Python version       : 3.14.0
IPython version      : 9.11.0

conda environment: n/a

Compiler    : Clang 20.1.4 
OS          : Darwin
Release     : 25.3.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

mgplot  : 0.2.21
pandas  : 3.0.1
pathlib : 1.0.1
requests: 2.32.5
typing  : 3.10.0.0
yfinance: 1.2.0

Watermark: 2.6.0



In [19]:
print("Finished")

Finished
